# 04 · Validate — the discrimination-problem analysis

**Standard slot:** *validate (in silico).* **For Project 05 this is the core science:** using the
hidden `truth` labels in the `EXAMPLE_DATA` pool, measure the **enrichment** each layer buys
(precision / recall), run a **cutoff-sensitivity sweep**, and show honestly that **no single metric
or layer cleanly separates true from false** (D3 part 2).

> Every number and figure here is `EXAMPLE_DATA` — **synthetic, for illustrating the method**, never a
> real result. On a real labeled pool (from a design paper with experimental outcomes) the same code
> produces a real analysis.

Needs `results/pool.csv` with the `truth` column.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Enrichment at each layer (precision & recall)

For each layer depth we ask: of the designs that survive to here, what fraction are *truly* good
(**precision** = enrichment), and what fraction of *all* truly-good designs do we still retain
(**recall**)? A good layer raises precision; watch the recall you pay for it. We run the layers and
read `layers_passed` against the planted `truth`.

In [ ]:
import filtering_pipeline as fp
import make_example_pool as mep
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

if not os.path.exists("results/pool.csv"):
    mep.make_pool(200).to_csv("results/pool.csv", index=False)
pool = pd.read_csv("results/pool.csv")

DESIGN_FIELDS = ["scrmsd", "plddt", "plddt_catalytic", "pae_interaction",
                 "catalytic_geom_rmsd", "scrmsd_orthogonal", "solubility",
                 "rosetta_dG", "shape_complementarity", "md_rmsd"]

def row_to_design(r):
    kw = {f: (None if pd.isna(r[f]) else float(r[f])) for f in DESIGN_FIELDS}
    return fp.Design(design_id=str(r["design_id"]), sequence="M",
                     design_type=str(r["design_type"]), extra={"truth": r["truth"]}, **kw)

# Run layers per design type (own cutoffs), then collect layers_passed + truth.
recs = []
for dt, sub in pool.groupby("design_type"):
    ds = [row_to_design(r) for _, r in sub.iterrows()]
    fp.run_pipeline(ds, design_type=dt, use_layers=(1, 2, 3, 4))
    for d in ds:
        recs.append(dict(design_id=d.design_id, design_type=dt,
                         layers_passed=d.layers_passed, truth=d.extra["truth"]))
res = pd.DataFrame(recs)
res["is_good"] = (res["truth"] == "good").astype(int)
n_good_total = int(res["is_good"].sum())
print(f"pool: {len(res)} designs, {n_good_total} truly-good (EXAMPLE_DATA labels)")

In [ ]:
rows = []
total = len(res)
for depth in range(0, 5):  # survivors that passed >= depth layers
    surv = res[res["layers_passed"] >= depth] if depth > 0 else res
    n = len(surv)
    n_good = int(surv["is_good"].sum())
    precision = n_good / n if n else float("nan")          # enrichment
    recall = n_good / n_good_total if n_good_total else float("nan")
    rows.append(dict(layer=("pool" if depth == 0 else f">=L{depth}"),
                     survivors=n, good_survivors=n_good,
                     precision=round(precision, 3), recall=round(recall, 3)))
enr = pd.DataFrame(rows)
print("Enrichment by layer depth (EXAMPLE_DATA):")
print(enr.to_string(index=False))

baseline = enr.iloc[0]["precision"]
print(f"\nbaseline good-fraction = {baseline:.3f}; "
      f"deepest-survivor precision = {enr.iloc[-1]['precision']:.3f} "
      f"(enrichment, but recall fell to {enr.iloc[-1]['recall']:.3f}).")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(enr["layer"], enr["precision"], "o-", label="precision (enrichment)")
ax[0].plot(enr["layer"], enr["recall"], "s--", label="recall")
ax[0].axhline(baseline, color="grey", ls=":", lw=0.8, label="baseline good-fraction")
ax[0].set_ylim(0, 1.02); ax[0].set_ylabel("fraction"); ax[0].set_title("Enrichment vs recall (EXAMPLE_DATA)")
ax[0].legend(fontsize=8)
ax[1].bar(enr["layer"], enr["survivors"]); ax[1].set_title("Survivors per layer (EXAMPLE_DATA)")
for i, v in enumerate(enr["survivors"]):
    ax[1].text(i, v, str(v), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig("results/p05_enrichment.png", dpi=150); plt.show()
print("saved results/p05_enrichment.png  (SYNTHETIC EXAMPLE_DATA)")

**The point:** precision rises layer by layer (the filter *enriches*) but never reaches 1.0 —
**false positives survive every layer** — and recall falls, so **true hits are discarded**. That gap
is the discrimination problem. On real data it is usually worse, not better.

## 2 · No single metric separates true from false

Plot each metric's distribution split by `truth`. The good/bad distributions **overlap** — there is no
threshold that cleanly separates them. (We show ROC-AUC per metric only as a separability summary; on
this synthetic pool it is illustrative, not a result.)

In [ ]:
from sklearn.metrics import roc_auc_score

metric_dir = {"scrmsd": -1, "plddt": +1, "pae_interaction": -1,
              "scrmsd_orthogonal": -1, "solubility": +1, "md_rmsd": -1}  # +1: higher=better
y = (pool["truth"] == "good").astype(int).values
print("Single-metric separability (ROC-AUC, EXAMPLE_DATA — illustrative only):")
aucs = {}
for m, direction in metric_dir.items():
    s = pool[m].values.astype(float) * direction
    mask = ~np.isnan(s)
    if len(set(y[mask])) > 1:
        aucs[m] = roc_auc_score(y[mask], s[mask])
        print(f"  {m:18s} AUC={aucs[m]:.3f}  (N={mask.sum()})")
print("\nNo metric reaches 1.0 — each only partially separates good from bad.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.5))
for ax, m in zip(axes.ravel(), metric_dir):
    g = pool.loc[pool["truth"] == "good", m].dropna()
    b = pool.loc[pool["truth"] == "bad", m].dropna()
    ax.hist(g, bins=20, alpha=0.6, label="good", density=True)
    ax.hist(b, bins=20, alpha=0.6, label="bad", density=True)
    ax.set_title(f"{m} (AUC={aucs.get(m, float('nan')):.2f})", fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle("Metric distributions by planted truth — note the OVERLAP (EXAMPLE_DATA)")
plt.tight_layout(); plt.savefig("results/p05_metric_overlap.png", dpi=150); plt.show()
print("saved results/p05_metric_overlap.png  (SYNTHETIC EXAMPLE_DATA)")

## 3 · Cutoff-sensitivity sweep

How brittle is the filter to its cutoffs? Sweep the Layer-1 `scrmsd` and `plddt` cutoffs over a
sensible range and watch **survival count** and **enrichment** move. Where the surface is flat, your
choice is safe; where it is steep, your short-list depends on an arbitrary threshold — say so.

In [ ]:
mono = pool[pool["design_type"] == "monomer"].copy()
yk = (mono["truth"] == "good").astype(int).values

scrmsd_grid = np.round(np.arange(1.5, 3.01, 0.25), 2)
plddt_grid = np.arange(70, 91, 5)

surv_grid = np.zeros((len(plddt_grid), len(scrmsd_grid)))
prec_grid = np.zeros_like(surv_grid)
for i, pl in enumerate(plddt_grid):
    for j, sc in enumerate(scrmsd_grid):
        keep = (mono["scrmsd"] <= sc) & (mono["plddt"] >= pl)
        n = int(keep.sum())
        surv_grid[i, j] = n
        prec_grid[i, j] = (yk[keep.values].mean() if n else np.nan)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
im0 = ax[0].imshow(surv_grid, origin="lower", aspect="auto", cmap="viridis")
ax[0].set_title("survivors (monomers, EXAMPLE_DATA)")
im1 = ax[1].imshow(prec_grid, origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=1)
ax[1].set_title("enrichment / precision (EXAMPLE_DATA)")
for a in ax:
    a.set_xticks(range(len(scrmsd_grid))); a.set_xticklabels(scrmsd_grid, fontsize=7)
    a.set_yticks(range(len(plddt_grid))); a.set_yticklabels(plddt_grid, fontsize=7)
    a.set_xlabel("scRMSD cutoff (<=)"); a.set_ylabel("pLDDT cutoff (>=)")
fig.colorbar(im0, ax=ax[0], fraction=0.046); fig.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.savefig("results/p05_cutoff_sensitivity.png", dpi=150); plt.show()
print("saved results/p05_cutoff_sensitivity.png  (SYNTHETIC EXAMPLE_DATA)")
print("Read it: tightening cutoffs raises enrichment but shrinks the survivor pool — the trade-off is the story.")

## 4 · Ranking stability under cutoff change `[extension]`

Does the **top-N** short-list change as cutoffs move? Compare the top-10 design IDs at a loose vs a
strict Layer-1 cutoff; the overlap (Jaccard) tells you how reproducible your short-list is. Low
overlap = your "best 10" is an artifact of an arbitrary threshold.

In [ ]:
def top_ids(scrmsd_cut, plddt_cut, k=10):
    ds = []
    for _, r in mono.iterrows():
        d = row_to_design(r)
        ds.append(d)
    cut = dict(scrmsd=scrmsd_cut, plddt=plddt_cut, pae=None)
    for d in ds:
        fp.self_consistency(d, cut)
    ranked = fp.rank_designs(ds)
    return [d.design_id for d in ranked[:k]]

loose = set(top_ids(3.0, 70))
strict = set(top_ids(1.8, 88))
jac = len(loose & strict) / len(loose | strict)
print(f"top-10 overlap (Jaccard) loose vs strict cutoffs: {jac:.2f}")
print("loose-only :", sorted(loose - strict)[:5], "...")
print("strict-only:", sorted(strict - loose)[:5], "...")
print("Low overlap ⇒ the short-list is cutoff-sensitive; report this, do not hide it.")

## D3 (part 2) checklist
- [ ] Enrichment (precision) + recall per layer on the labeled pool; the gap from 1.0 discussed.
- [ ] Metric-overlap figure: no single metric separates good/bad (overlapping distributions).
- [ ] Cutoff-sensitivity sweep (survival + enrichment surfaces); brittle regions identified.
- [ ] Ranking-stability check (top-N overlap under cutoff change).
- [ ] Every figure labeled `EXAMPLE_DATA`; false positives among survivors stated explicitly.

**Next:** `05_validation_plan.ipynb` — optional Layer 4 (MD) hook, packaging, and cohort adoption.